In [1]:
import sys, os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
print(sys.executable)
print(os.environ.get("JAVA_HOME"))

/bin/python
/usr/lib/jvm/java-21-openjdk


In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 00:15:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
spark.version

'4.1.1'

In [5]:
!ls -lh yellow_tripdata_2025-11.parquet

-rw-r--r--. 1 miko miko 68M 12-19 16:51 yellow_tripdata_2025-11.parquet


In [6]:
schema = types.StructType([
    types.StructField('VendorID', types.IntegerType(), True),
    types.StructField('tpep_pickup_datetime', types.TimestampType(), True),
    types.StructField('tpep_dropoff_datetime', types.TimestampType(), True),
    types.StructField('passenger_count', types.IntegerType(), True),
    types.StructField('trip_distance', types.DoubleType(), True),
    types.StructField('RatecodeID', types.IntegerType(), True),
    types.StructField('store_and_fwd_flag', types.StringType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('payment_type', types.IntegerType(), True),
    types.StructField('fare_amount', types.DoubleType(), True),
    types.StructField('extra', types.DoubleType(), True),
    types.StructField('mta_tax', types.DoubleType(), True),
    types.StructField('tip_amount', types.DoubleType(), True),
    types.StructField('tolls_amount', types.DoubleType(), True),
    types.StructField('improvement_surcharge', types.DoubleType(), True),
    types.StructField('total_amount', types.DoubleType(), True),
    types.StructField('congestion_surcharge', types.DoubleType(), True)
])

In [9]:
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')

df.repartition(4).write.parquet('homework_pq', mode='overwrite')

In [ ]:
df = spark.read.parquet('homework_pq')

**Q3**: How many taxi trips were there on February 15?

In [11]:
from pyspark.sql import functions as F

In [13]:
df \
    .withColumn('tpep_pickup_datetime', F.to_date(df.tpep_pickup_datetime)) \
    .filter("tpep_pickup_datetime = '2025-11-15'") \
    .count()

162604

In [15]:
df.createOrReplaceTempView('yellow_2025_11')

In [16]:
spark.sql("""
SELECT
    COUNT(1)
FROM 
    yellow_2025_11
WHERE
    to_date(tpep_pickup_datetime) = '2025-11-15';
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



**Q4**: What is the length of the longest trip in the dataset in hours?

In [19]:
df.withColumn('duration_hrs', 
    (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 3600) \
    .select(F.max('duration_hrs')) \
    .show()

+-----------------+
|max(duration_hrs)|
+-----------------+
|90.64666666666666|
+-----------------+



**Q6**: Least frequent pickup location zone

In [23]:
df_zones = spark.read.option("header", "true").csv('taxi_zone_lookup.csv')

In [24]:
df_zones.columns

['LocationID', 'Borough', 'Zone', 'service_zone']

In [25]:
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [26]:
df_zones.createOrReplaceTempView('zones')

In [28]:
spark.sql("""
SELECT
    pul.Zone,
    COUNT(1)
FROM 
    yellow_2025_11 y LEFT JOIN zones pul ON y.PULocationID = pul.LocationID
GROUP BY 
    1
ORDER BY
    2 ASC
LIMIT 5;
""").show()

+--------------------+--------+
|                Zone|count(1)|
+--------------------+--------+
|Governor's Island...|       1|
|Eltingville/Annad...|       1|
|       Arden Heights|       1|
|       Port Richmond|       3|
|       Rikers Island|       4|
+--------------------+--------+

